#### واردات کتابخانه ها

In [27]:
import pandas as pd
import numpy as np
from hazm import Normalizer, WordTokenizer
import re
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import FastText, Word2Vec
from transformers import AutoTokenizer, AutoModel
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

#### بارگذاری دیتاست (آموزشی)

In [2]:
df = pd.read_parquet("./data/sentiment_dksf_train.parquet")

In [3]:
print("ستون‌های دیتاست :", df.columns.tolist())

ستون‌های دیتاست : ['text', 'label']


In [4]:
print("توزیع کلاس های ستون\n", df['label'].value_counts())

توزیع کلاس های ستون
 label
0    10206
1     9587
2     8809
Name: count, dtype: int64


In [5]:
print("تعداد نمونه‌ها :", len(df))

تعداد نمونه‌ها : 28602


In [6]:
print("نمونه‌های اولیه :")
display(df.head(5))

نمونه‌های اولیه :


,text,label
0,شیشه هاش همون روز اول شکست,0
1,تمیز -مرتب- چقدر پیک آن تایم و مؤدب …سپاس بسیا...,1
2,نظر من هیچ گاه این محصول رو خریداری نکنید اگه ...,0
3,برنج ایرانی سفارش دادم افتضاح بود با هندی هیچ ...,0
4,سلام اصلا راضی نیستم یعنی ارزش خرید نداره,0


#### آمار متن قبل از پیش پردازش

In [7]:
def text_stats(df, text_col='text'):
    lengths_char = df[text_col].str.len()
    lengths_word = df[text_col].apply(lambda x: len(x.split()))
    unique_words = len(set(' '.join(df[text_col].astype(str)).split()))
    
    print(f"میانگین طول متن (کاراکتر) : {lengths_char.mean():.2f}")
    print(f"میانگین طول متن (کلمه) : {lengths_word.mean():.2f}")
    print(f"تعداد کلمات منحصربه‌فرد : {unique_words}")

print("آمار متن قبل از پیش پردازش :")
text_stats(df)

آمار متن قبل از پیش پردازش :
میانگین طول متن (کاراکتر) : 125.53
میانگین طول متن (کلمه) : 25.77
تعداد کلمات منحصربه‌فرد : 58975


#### پیش پردازش متن

In [8]:
normalizer = Normalizer()

tokenizer = WordTokenizer()

custom_stopwords = {
    # حروف اضافه و ربط خیلی رایج
    'از', 'به', 'در', 'با', 'برای', 'تا', 'را', 'و', 'که', 'این', 'اون', 'آن', 'های', 'ها', 'ای', 'ام', 'ات', 'اش', 'شان', 'تان', 'مان',

    # افعال کمکی و بودن/شدن
    'است', 'بود', 'بودم', 'بودی', 'بودیم', 'بودند', 'هست', 'هستم', 'هستی', 'هستیم', 'هستند', 'شد', 'شدم', 'شدی', 'شدیم', 'شدند',

    # کلمات خیلی عمومی بدون بار احساسی قوی
    'می', 'میشه', 'میتونه', 'میتونم', 'میتونی', 'میکنه', 'میکنم', 'میکنی', 'میکنیم', 'میکنن',
    'یه', 'ی', 'هم', 'نیز', 'یا', 'ولی', 'اما', 'اگر', 'تا', 'پس', 'چون', 'هر', 'همه', 'هیچ', 'چند', 'بسیار', 'خیلی', 'کمی',

    # ضمایر و کلمات اشاره‌ای خیلی رایج
    'من', 'تو', 'او', 'ما', 'شما', 'ایشان', 'خود', 'خودم', 'خودت', 'خودش', 'خودمون', 'خودتون',

    # برخی کلمات ساختاری
    'بودن', 'شدن', 'کردن', 'داشتن', 'داشتم', 'داشتی', 'داشتیم', 'داشتند'
}

stopwords = custom_stopwords

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^\w\s\u0600-\u06FF]', '', text)
    text = normalizer.normalize(text)
    text = text.replace('\u200c', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_text(text):
    cleaned = clean_text(text)
    if not cleaned:
        return []
    tokens = tokenizer.tokenize(cleaned)
    tokens = [
        t for t in tokens
        if t not in stopwords
        and len(t) > 1
        and not t.isdigit()
    ]
    return tokens

In [9]:
print("...در حال پیش پردازش متن‌ها (ممکن است چند دقیقه طول بکشه)")
df['clean_text'] = df['text'].apply(clean_text)
df['tokens'] = df['clean_text'].apply(preprocess_text)
df['clean_text_joined'] = df['tokens'].apply(lambda x: ' '.join(x) if x else '')

...در حال پیش پردازش متن‌ها (ممکن است چند دقیقه طول بکشه)


#### آمار متن بعد از پیش پردازش

In [10]:
print("آمار متن بعد از پیش پردازش :")
text_stats(df, text_col='clean_text_joined')

آمار متن بعد از پیش پردازش :
میانگین طول متن (کاراکتر) : 98.21
میانگین طول متن (کلمه) : 18.31
تعداد کلمات منحصربه‌فرد : 43449


In [11]:
print("نمونه های قبل و بعد پیش پردازش :")

df_display = df[['text', 'clean_text', 'tokens']].head(5).copy()
df_display['tokens'] = df_display['tokens'].apply(lambda x: x[:15] if isinstance(x, (list, tuple)) else x)

df_display = df_display.rename(columns={
    'text': 'متن خام',
    'clean_text': 'متن تمیز شده',
    'tokens': 'توکن‌ها (15 تای اول)'
})

display(df_display.style.hide(axis='index'))

نمونه های قبل و بعد پیش پردازش :


متن خام,متن تمیز شده,توکن‌ها (15 تای اول)
شیشه هاش همون روز اول شکست,شیشه هاش همون روز اول شکست,"['شیشه', 'هاش', 'همون', 'روز', 'اول', 'شکست']"
تمیز -مرتب- چقدر پیک آن تایم و مؤدب …سپاس بسیار عالی فقط کاشکی رنگ دستمال همونی می‌بود که در شکل انتخاب کردیم … باز هم سپاس بسیار عالی از سرعت و کیفیت,تمیز مرتب چقدر پیک آن تایم و مؤدب سپاس بسیار عالی فقط کاشکی رنگ دستمال همونی می بود که در شکل انتخاب کردیم باز هم سپاس بسیار عالی از سرعت و کیفیت,"['تمیز', 'مرتب', 'چقدر', 'پیک', 'تایم', 'مؤدب', 'سپاس', 'عالی', 'فقط', 'کاشکی', 'رنگ', 'دستمال', 'همونی', 'شکل', 'انتخاب']"
نظر من هیچ گاه این محصول رو خریداری نکنید اگه می خواهید شانستون رو امتحان کنین ببینین کارت هاتون 108 تا یا 52 هستش راه باز هست بعضی ها میگن 108 کارت داره بعضی ها هم میگن 54 تا کارت داره در اصل این محصول باید 108 کارت باشه که برای من 52 تا دراومد و فارسی هم هستش و برای یک کالای دیگه هستش که قیمتش ارزونتره,نظر من هیچ گاه این محصول رو خریداری نکنید اگه می خواهید شانستون رو امتحان کنین ببینین کارت هاتون ۱۰۸ تا یا ۵۲ هستش راه باز هست بعضی ها میگن ۱۰۸ کارت داره بعضی ها هم میگن ۵۴ تا کارت داره در اصل این محصول باید ۱۰۸ کارت باشه که برای من ۵۲ تا دراومد و فارسی هم هستش و برای یک کالای دیگه هستش که قیمتش ارزونتره,"['نظر', 'گاه', 'محصول', 'رو', 'خریداری', 'نکنید', 'اگه', 'خواهید_شانستون', 'رو', 'امتحان', 'کنین', 'ببینین', 'کارت', 'هاتون', 'هستش']"
برنج ایرانی سفارش دادم افتضاح بود با هندی هیچ فرقی نمیکرد,برنج ایرانی سفارش دادم افتضاح بود با هندی هیچ فرقی نمی کرد,"['برنج', 'ایرانی', 'سفارش', 'دادم', 'افتضاح', 'هندی', 'فرقی', 'نمی', 'کرد']"
سلام اصلا راضی نیستم یعنی ارزش خرید نداره,سلام اصلا راضی نیستم یعنی ارزش خرید نداره,"['سلام', 'اصلا', 'راضی', 'نیستم', 'یعنی', 'ارزش', 'خرید', 'نداره']"


#### NaN بررسی تعداد داده های

In [12]:
print("clean_text_joined در NaN تعداد :\n", df['clean_text_joined'].isna().sum())
print("\nتعداد رشته خالی ('') :", (df['clean_text_joined'] == '').sum())

problematic = df[df['clean_text_joined'].isna() | (df['clean_text_joined'] == '')]

print("\nنمونه ردیف‌های مشکل‌دار (5 مورد اول) :")
df_problem = problematic[['text', 'clean_text', 'clean_text_joined']].head(5)
df_problem = df_problem.rename(columns={
        'text': 'متن اصلی',
        'clean_text': 'متن تمیز شده',
        'clean_text_joined': '(joined) متن نهایی'
})

display(df_problem)

print("\nتعداد کل ردیف‌های مشکل‌دار :", len(problematic))

clean_text_joined در NaN تعداد :
 0

تعداد رشته خالی ('') : 161

نمونه ردیف‌های مشکل‌دار (5 مورد اول) :


,متن اصلی,متن تمیز شده,(joined) متن نهایی
79,.............,,
255,۴و ۳,۴ و ۳,
360,۲٫ ۳٫۸,۲ ٫ ۳ ٫ ۸,
490,و۲و۸,و ۲ و ۸,
775,--,,



تعداد کل ردیف‌های مشکل‌دار : 161


#### NaN حذف داده های

In [13]:
print("قبل از حذف :")
print("NaN تعداد :", df['clean_text_joined'].isna().sum())
print("تعداد کل نمونه‌ها قبل :", len(df))

df['clean_text_joined'] = df['clean_text_joined'].fillna('')
df['clean_text_joined'] = df['clean_text_joined'].astype(str)
df = df[df['clean_text_joined'].str.strip() != ''].reset_index(drop=True)

print("\nبعد از حذف :")
print("تعداد نمونه باقی‌مانده :", len(df))
print("NaN تعداد :", df['clean_text_joined'].isna().sum())

print("\nبررسی نهایی :")
print("تعداد رشته خالی باقی‌مانده :", (df['clean_text_joined'] == '').sum())
print("strip حداقل طول متن بعد از :\n", df['clean_text_joined'].str.strip().str.len().min())

print("\nclean_text_joined نوع داده :")
print(df['clean_text_joined'].apply(type).value_counts())

قبل از حذف :
NaN تعداد : 0
تعداد کل نمونه‌ها قبل : 28602

بعد از حذف :
تعداد نمونه باقی‌مانده : 28441
NaN تعداد : 0

بررسی نهایی :
تعداد رشته خالی باقی‌مانده : 0
strip حداقل طول متن بعد از :
 2

clean_text_joined نوع داده :
clean_text_joined
<class 'str'>    28441
Name: count, dtype: int64


In [14]:
df.to_csv("./data/sentiment_processed.csv", index=False, encoding='utf-8-sig')
print(".انجام شد sentiment_processed.csv ذخیره")

.انجام شد sentiment_processed.csv ذخیره


In [15]:
df = pd.read_csv("./data/sentiment_processed.csv")

In [16]:
print("ستون‌ها :", df.columns.tolist())

ستون‌ها : ['text', 'label', 'clean_text', 'tokens', 'clean_text_joined']


#### نمایش متن

<div align="center">TF-IDF</div>

#### تصمیم گیری برای بررسی تعداد ویژگی ها

In [17]:
for mf in [5000, 10000, 15000, None]:
    print(f"\n--- max_features = {mf} ---")
    vec = TfidfVectorizer(max_features=mf, min_df=2, analyzer='word')
    emb = vec.fit_transform(df['clean_text_joined'])
    vocab_size = len(vec.vocabulary_)
    all_unique = len(set(' '.join(df['clean_text_joined']).split()))
    coverage = vocab_size / all_unique * 100 if all_unique else 0
    print(f"ویژگی‌ها: {vocab_size:,} | پوشش: %{coverage:.2f}")
    print(f"ابعاد: {emb.shape}")


--- max_features = 5000 ---
ویژگی‌ها: 5,000 | پوشش: %11.51
ابعاد: (28441, 5000)

--- max_features = 10000 ---
ویژگی‌ها: 10,000 | پوشش: %23.02
ابعاد: (28441, 10000)

--- max_features = 15000 ---
ویژگی‌ها: 15,000 | پوشش: %34.52
ابعاد: (28441, 15000)

--- max_features = None ---
ویژگی‌ها: 16,307 | پوشش: %37.53
ابعاد: (28441, 16307)


In [28]:
vectorizer = TfidfVectorizer(max_features=15000, min_df=2, analyzer='word')
tfidf_embeddings = vectorizer.fit_transform(df['clean_text_joined'])

In [29]:
joblib.dump(vectorizer, r"./data/tfidf_vectorizer.pkl")
print(".فیت و ذخیره شد train روی Vectorizer")

.فیت و ذخیره شد train روی Vectorizer


In [ ]:
vectorizer = joblib.load(r".\data\tfidf_vectorizer.pkl")
print(".لود شد Vectorizer")

In [30]:
print("TF-IDF Embeddings :")
print(f"ابعاد ماتریس : {tfidf_embeddings.shape}")
print(f"تعداد واژگان پوشش داده‌شده : {len(vectorizer.vocabulary_)}")
print("برای 10 متن اول embedding نمونه :")
print(tfidf_embeddings[0].toarray()[0][:10])

all_words = ' '.join(df['clean_text_joined']).split()
unique_words = set(all_words)
tfidf_coverage = len(vectorizer.vocabulary_) / len(unique_words) * 100 if unique_words else 0
print(f"پوشش واژگان : %{tfidf_coverage:.2f}")

TF-IDF Embeddings :
ابعاد ماتریس : (28441, 15000)
تعداد واژگان پوشش داده‌شده : 15000
برای 10 متن اول embedding نمونه :
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
پوشش واژگان : %34.52


In [19]:
np.save(r"D:\data-Project-NLP\tfidf_embeddings.npy", tfidf_embeddings.toarray())
print("\n.انجام شد tfidf_embeddings.npy ذخیره")


.انجام شد tfidf_embeddings.npy ذخیره


In [18]:
tfidf_embeddings = np.load(r"D:\data-Project-NLP\tfidf_embeddings.npy")

<div align="center">Fast Text</div>

In [21]:
sentences = df['tokens'].tolist()

fasttext_model = FastText(
    sentences=sentences,
    vector_size=300,
    window=5,
    min_count=5,
    epochs=20,
    sg=1,
    workers=4
)

def get_fasttext_embedding(tokens, model):
    vectors = [model.wv[t] for t in tokens if t in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

fasttext_embeddings = np.array([get_fasttext_embedding(tokens, fasttext_model) for tokens in df['tokens']])

print("FastText Embeddings :")
print(f"ابعاد هر بردار : {fasttext_model.vector_size}")
print(f"ابعاد ماتریس : {fasttext_embeddings.shape}")
print(f"تعداد واژگان پوشش داده‌شده : {len(fasttext_model.wv)}")
print("برای متن اول embedding نمونه :")
print(fasttext_embeddings[0][:10])

fasttext_coverage = len(fasttext_model.wv) / len(unique_words) * 100 if unique_words else 0
print(f"پوشش واژگان : %{fasttext_coverage:.2f}")

FastText Embeddings :
ابعاد هر بردار : 300
ابعاد ماتریس : (28441, 300)
تعداد واژگان پوشش داده‌شده : 104
برای متن اول embedding نمونه :
[ 0.02743842  0.00723743 -0.00403445 -0.00122524 -0.02884895 -0.01821702
  0.01248218  0.00263962 -0.0028816   0.02043389]
پوشش واژگان : %0.24


In [22]:
sentences = df['tokens'].tolist()

fasttext_model = FastText(
    sentences=sentences,
    vector_size=300,
    window=5,
    min_count=2,
    epochs=25,
    sg=1,
    workers=4
)

def get_fasttext_embedding(tokens, model):
    vectors = [model.wv[t] for t in tokens if t in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

fasttext_embeddings = np.array([get_fasttext_embedding(tokens, fasttext_model) for tokens in df['tokens']])

print("FastText Embeddings :")
print(f"ابعاد هر بردار : {fasttext_model.vector_size}")
print(f"ابعاد ماتریس : {fasttext_embeddings.shape}")
print(f"تعداد واژگان پوشش داده‌شده : {len(fasttext_model.wv)}")
print("برای متن اول embedding نمونه :")
print(fasttext_embeddings[0][:10])

fasttext_coverage = len(fasttext_model.wv) / len(unique_words) * 100 if unique_words else 0
print(f"پوشش واژگان : %{fasttext_coverage:.2f}")

FastText Embeddings :
ابعاد هر بردار : 300
ابعاد ماتریس : (28441, 300)
تعداد واژگان پوشش داده‌شده : 107
برای متن اول embedding نمونه :
[ 0.04484306  0.00432509 -0.01446144 -0.01642148  0.02709966 -0.03784044
  0.03254221  0.04056868  0.00640705  0.04165835]
پوشش واژگان : %0.25


In [23]:
np.save(r"D:\data-Project-NLP\fasttext_embeddings.npy", fasttext_embeddings)
print("\n.انجام شد fasttext_embeddings.npy ذخیره")


.انجام شد fasttext_embeddings.npy ذخیره


In [19]:
fasttext_embeddings = np.load(r"D:\data-Project-NLP\fasttext_embeddings.npy")

<div align="center">Word2Vec</div>

In [29]:
sentences = df['tokens'].tolist()

word2vec_model = Word2Vec(
    sentences=sentences,
    vector_size=300,
    window=5,
    min_count=3,
    epochs=25,
    sg=1,
    workers=4,
    negative=5
)

def get_word2vec_embedding(tokens, model):
    vectors = [model.wv[t] for t in tokens if t in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

word2vec_embeddings = np.array([
    get_word2vec_embedding(tokens, word2vec_model) 
    for tokens in df['tokens']
])

print("Word2Vec Embeddings :")
print(f"ابعاد هر بردار : {word2vec_model.vector_size}")
print("embeddings ابعاد ماتریس :")
print(word2vec_embeddings.shape)
print(f"تعداد واژگان مدل : {len(word2vec_model.wv):,}")

all_unique_words = len(set(' '.join(df['clean_text_joined']).split()))
coverage = len(word2vec_model.wv) / all_unique_words * 100 if all_unique_words > 0 else 0
print(f"پوشش واژگان تقریبی : %{coverage:.2f}")

print("متن اول (۱۰ بعد اول) embedding نمونه :")
print(word2vec_embeddings[0][:10])

Word2Vec Embeddings :
ابعاد هر بردار : 300
embeddings ابعاد ماتریس :
(28441, 300)
تعداد واژگان مدل : 106
پوشش واژگان تقریبی : %0.24
متن اول (۱۰ بعد اول) embedding نمونه :
[-0.00152851 -0.02327841 -0.03634987  0.05699404  0.101305   -0.03872554
  0.07237802  0.01871239 -0.0081173   0.03536129]


In [27]:
np.save(r"D:\data-Project-NLP\word2vec_embeddings.npy", word2vec_embeddings)
print("\n.انجام شد word2vec_embeddings.npy ذخیره")


.انجام شد word2vec_embeddings.npy ذخیره


In [20]:
word2vec_embeddings = np.load(r"D:\data-Project-NLP\word2vec_embeddings.npy")

<div align="center">ParsBERT</div>

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"استفاده از دستگاه : {device}")

model_name = "HooshvareLab/bert-fa-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

def get_parsbert_batch(texts, batch_size=16, max_length=128):
    embeddings = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                max_length=max_length,
                truncation=True,
                padding=True
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)
            batch_emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            embeddings.append(batch_emb)
            print(f"Batch {i//batch_size + 1} از {len(texts)//batch_size + 1} پردازش شد")
    return np.vstack(embeddings)

texts = df['clean_text_joined'].tolist()
print(f"\nتعداد متن‌ها: {len(texts)}")

print("...(ممکن است چند دقیقه طول بکشد) ParsBERT embeddings در حال محاسبه")
parsbert_embeddings = get_parsbert_batch(texts, batch_size=16)

print("\nParsBERT Embeddings :")
print(f"ابعاد هر  بردار : {parsbert_embeddings.shape[1]}")
print(f"ابعاد ماتریس : {parsbert_embeddings.shape}")
print("tokenizer اندازه واژگان :")
print(len(tokenizer.vocab))
print("برای متن اول embedding نمونه :")
print(parsbert_embeddings[0][:10])

coverage_pb = 100.0
print(f"پوشش واژگان تقریبی : %{coverage_pb:.2f}")

استفاده از دستگاه : cpu


config.json:   0%|          | 0.00/440 [00:00<?, ?B/s]

c:\Users\amir_v_z\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\amir_v_z\.cache\huggingface\hub\models--HooshvareLab--bert-fa-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]


تعداد متن‌ها: 28441
...(ممکن است چند دقیقه طول بکشد) ParsBERT embeddings در حال محاسبه
Batch 1 از 1778 پردازش شد
Batch 2 از 1778 پردازش شد
Batch 3 از 1778 پردازش شد
Batch 4 از 1778 پردازش شد
Batch 5 از 1778 پردازش شد
Batch 6 از 1778 پردازش شد
Batch 7 از 1778 پردازش شد
Batch 8 از 1778 پردازش شد
Batch 9 از 1778 پردازش شد
Batch 10 از 1778 پردازش شد
Batch 11 از 1778 پردازش شد
Batch 12 از 1778 پردازش شد
Batch 13 از 1778 پردازش شد
Batch 14 از 1778 پردازش شد
Batch 15 از 1778 پردازش شد
Batch 16 از 1778 پردازش شد
Batch 17 از 1778 پردازش شد
Batch 18 از 1778 پردازش شد
Batch 19 از 1778 پردازش شد
Batch 20 از 1778 پردازش شد
Batch 21 از 1778 پردازش شد
Batch 22 از 1778 پردازش شد
Batch 23 از 1778 پردازش شد
Batch 24 از 1778 پردازش شد
Batch 25 از 1778 پردازش شد
Batch 26 از 1778 پردازش شد
Batch 27 از 1778 پردازش شد
Batch 28 از 1778 پردازش شد
Batch 29 از 1778 پردازش شد
Batch 30 از 1778 پردازش شد
Batch 31 از 1778 پردازش شد
Batch 32 از 1778 پردازش شد
Batch 33 از 1778 پردازش شد
Batch 34 از 1778 پردازش شد
Bat

In [5]:
np.save(r"D:\data-Project-NLP\parsbert_embeddings.npy", parsbert_embeddings)
print("\n.انجام شد parsbert_embeddings.npy ذخیره")


.انجام شد parsbert_embeddings.npy ذخیره


In [21]:
parsbert_embeddings = np.load(r"D:\data-Project-NLP\parsbert_embeddings.npy")

#### مقایسه روش ها

In [22]:
def compute_similarity(emb1, emb2):
    emb1 = np.asarray(emb1).flatten()
    emb2 = np.asarray(emb2).flatten()
    return cosine_similarity([emb1], [emb2])[0][0]

sim_tfidf    = compute_similarity(tfidf_embeddings[0], tfidf_embeddings[1])
sim_word2vec = compute_similarity(word2vec_embeddings[0], word2vec_embeddings[1])
sim_fasttext = compute_similarity(fasttext_embeddings[0], fasttext_embeddings[1])
sim_parsbert = compute_similarity(parsbert_embeddings[0], parsbert_embeddings[1])

all_unique_words = len(set(' '.join(df['clean_text_joined']).split())) if 'df' in globals() else 1

tfidf_cov   = len(vectorizer.vocabulary_) / all_unique_words * 100 if 'vectorizer' in globals() else 34.05
word2vec_cov = (len(word2vec_model.wv) / all_unique_words * 100 
                if 'word2vec_model' in globals() and len(word2vec_model.wv) > 0 else 0.24)
fasttext_cov = fasttext_coverage if 'fasttext_coverage' in globals() else 0.25
parsbert_cov = 100.0

comparison = pd.DataFrame({
    "روش": ["TF-IDF", "Word2Vec", "FastText", "ParsBERT"],
    
    "ابعاد بردار": [
        tfidf_embeddings.shape[1],
        word2vec_embeddings.shape[1],
        fasttext_embeddings.shape[1],
        parsbert_embeddings.shape[1]
    ],
    
    "درصد پوشش واژگان تقریبی": [
        tfidf_cov,
        word2vec_cov,
        fasttext_cov,
        parsbert_cov
    ],
    
    "پارامترهای کلیدی": [
        "max_features=15000, min_df=2, analyzer='word'",
        "vector_size=300, window=5, min_count=3, epochs=25, sg=1, negative=5",
        "vector_size=300, window=5, min_count=2~5, epochs=20~25, sg=1",
        "hidden_size=768, layers=12, mean pooling, max_length=128"
    ],
    
    "شباهت متن ۰ در مقابل متن ۱": [
        sim_tfidf,
        sim_word2vec,
        sim_fasttext,
        sim_parsbert
    ]
})

print("\n(Embedding) مقایسه روش‌های نمایش متن :")
display(comparison)

print("\nجدول مرتب‌شده بر اساس پوشش واژگان (نزولی) :")
display(comparison.sort_values("درصد پوشش واژگان تقریبی", ascending=False))


(Embedding) مقایسه روش‌های نمایش متن :


,روش,ابعاد بردار,درصد پوشش واژگان تقریبی,پارامترهای کلیدی,شباهت متن ۰ در مقابل متن ۱
0,TF-IDF,15000,34.05,"max_features=15000, min_df=2, analyzer='word'",0.000000
1,Word2Vec,300,0.24,"vector_size=300, window=5, min_count=3, epochs...",0.993856
2,FastText,300,0.25,"vector_size=300, window=5, min_count=2~5, epoc...",0.987243
3,ParsBERT,768,100.00,"hidden_size=768, layers=12, mean pooling, max_...",0.631646



جدول مرتب‌شده بر اساس پوشش واژگان (نزولی) :


,روش,ابعاد بردار,درصد پوشش واژگان تقریبی,پارامترهای کلیدی,شباهت متن ۰ در مقابل متن ۱
3,ParsBERT,768,100.00,"hidden_size=768, layers=12, mean pooling, max_...",0.631646
0,TF-IDF,15000,34.05,"max_features=15000, min_df=2, analyzer='word'",0.000000
2,FastText,300,0.25,"vector_size=300, window=5, min_count=2~5, epoc...",0.987243
1,Word2Vec,300,0.24,"vector_size=300, window=5, min_count=3, epochs...",0.993856


#### بارگذاری دیتاست (تستی)

In [23]:
df_test = pd.read_parquet("./data/sentiment_dksf_test.parquet")

In [24]:
print("داده تست لود شد")
print("تعداد ردیف‌ها :", len(df_test))
print("ستون‌ها :", df_test.columns.tolist())
print("\nنمونه 5 ردیف اول :")
display(df_test[['text', 'label']].head(5))

داده تست لود شد
تعداد ردیف‌ها : 2315
ستون‌ها : ['text', 'label']

نمونه 5 ردیف اول :


,text,label
0,عطره و بیشتر به درد لباس میخوره، چون اصلا پخش ...,0
1,اصلا بدرد نمیخوره.بعد از دو سه روز کنده میشه.ز...,0
2,غذا بسیار با کیفیت بود اما متاسفانه دوتا سفارش...,1
3,غذا خیلی خوب بود، خیلی سریع رسید و قیمتشم خیلی...,1
4,قیمت رو بالا ببرید کیفیت رو کم نکنید لطفا,1


In [25]:
df_test['clean_text'] = df_test['text'].apply(clean_text)
df_test['tokens'] = df_test['clean_text'].apply(preprocess_text)
df_test['clean_text_joined'] = df_test['tokens'].apply(lambda x: ' '.join(x) if x else '')

df_test = df_test[df_test['clean_text_joined'].str.strip() != ''].reset_index(drop=True)

print("داده تست بعد از پیش پردازش :")
print("تعداد ردیف‌ها :", len(df_test))
print("clean_text_joined در NaN تعداد :\n", df_test['clean_text_joined'].isna().sum())
print("تعداد خالی :", (df_test['clean_text_joined'] == '').sum())

داده تست بعد از پیش پردازش :
تعداد ردیف‌ها : 2312
clean_text_joined در NaN تعداد :
 0
تعداد خالی : 0


In [26]:
df.to_csv("./data/sentiment_processed_test.csv", index=False, encoding='utf-8-sig')
print(".انجام شد sentiment_processed_test.csv ذخیره")

.انجام شد sentiment_processed_test.csv ذخیره


In [ ]:
df_test = pd.read_csv("./data/sentiment_processed_test.csv")

In [33]:
X_tfidf_test = vectorizer.transform(df_test['clean_text_joined'])
print(": تست ساخته شد. ابعاد TF-IDF\n", X_tfidf_test.shape)

: تست ساخته شد. ابعاد TF-IDF
 (2312, 15000)


In [ ]:
texts_test = df_test['clean_text_joined'].tolist()
parsbert_embeddings_test = get_parsbert_batch(texts_test, batch_size=16)
print(": تست ساخته شد. ابعاد ParsBERT\n", parsbert_embeddings_test.shape)